In [ ]:
# Допустим, это модель предсказания вероятности того, что бренд одежды понравится пользователю интернет-магазина :)
import torch
from torch.optim import SGD

torch.manual_seed(42)

# Датасет: 9 параметров на бренд, последний элемент высутпает целевым показателем
dataset = [
    (torch.tensor([1, 0, 0.7, 0.8, 0.6, 0.7, 0.9, 0.5, 0.8]), torch.tensor([1.0])), # Пример: бренд с фокусом на классику
    (torch.tensor([0, 1, 0.4, 0.7, 0.9, 0.8, 0.7, 0.4, 0.6]), torch.tensor([1.0])), # Пример: модный и молодежный бренд
    (torch.tensor([1, 0, 0.5, 0.6, 0.5, 0.6, 0.8, 0.6, 0.7]), torch.tensor([0.0])), # Пример: масс-маркет бренд
    (torch.tensor([0, 1, 0.3, 0.4, 0.8, 0.7, 0.6, 0.3, 0.5]), torch.tensor([0.0])), # Пример: нишевый/агрессивный уличный бренд
    (torch.tensor([1, 1, 0.9, 0.9, 0.7, 0.9, 0.9, 0.8, 0.9]), torch.tensor([1.0])), # Пример: премиальный/дорогой бренд
    (torch.tensor([0, 0, 0.2, 0.3, 0.4, 0.5, 0.5, 0.2, 0.3]), torch.tensor([0.0])), # Пример: бренд повседневной одежды ака смарт кэжуал
    (torch.tensor([1, 0, 0.6, 0.8, 0.5, 0.6, 0.7, 0.5, 0.6]), torch.tensor([1.0])), # Пример: пижама
    (torch.tensor([0, 1, 0.3, 0.5, 0.6, 0.5, 0.6, 0.3, 0.4]), torch.tensor([0.0])), # Пример: монашеские одеяния
    (torch.tensor([1, 1, 0.8, 0.9, 0.9, 0.8, 0.9, 0.7, 0.8]), torch.tensor([1.0]))  # Пример: водолазные костюмы
]

# Три группы весов: внешний вид, эвосприятие, контекстуальные данные
w_visual = torch.rand((1, 3), requires_grad=True)  # Веса для параметров: стиль, цветовая гамма, материал
w_emotional = torch.rand((1, 3), requires_grad=True)  # Веса для восприятия: трендовость, оригинальность, удобство
w_contextual = torch.rand((1, 3), requires_grad=True)   # Веса для популярности, ценовой категории, доступности

# Устанавливаем bias, он же смещение
bias = torch.tensor([0.5], requires_grad=True)

# Оптимизатор, реализовнный за счет градиентного спуска
optimizer = SGD([w_visual, w_emotional, w_contextual, bias], lr=0.01)

# Функция предсказания
def predict_likelihood(brand_features: torch.Tensor) -> torch.Tensor:
    visual = brand_features[2:5] @ w_visual.T
    emotional = brand_features[5:8] @ w_emotional.T
    contextual = (
        brand_features[:2].unsqueeze(0) @ w_contextual[:, :2].T
        + brand_features[8:] @ w_contextual[:, 2:].T
    )
    return visual + emotional + contextual + bias

# Функция потерь
def loss_function(y_pred, y_actual):
    return torch.nn.functional.mse_loss(y_pred, y_actual)

# Обучение модели
for epoch in range(100):  # ставим 100 эпох обучения
    total_loss = 0
    for brand, label in dataset:
        optimizer.zero_grad()
        predicted = predict_likelihood(brand) 
        loss = loss_function(predicted, label) 
        total_loss += loss.item()
        loss.backward() # Обратное распространение ошибки
        optimizer.step()

    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch + 1}, Loss: {total_loss:.4f}")

print("\nFinal Weights:")
print("Visual Weights:", w_visual)
print("Emotional Weights:", w_emotional)
print("Contextual Weights:", w_contextual)
print("Bias:", bias)
